<h1>📘 Biofilter — Reports 101 (4.3.0)</h1>

How reports work now that they read a **bundle** instead of a database.

A bundle is a folder: parquet files plus a `manifest.json` that says what
is in them. It is immutable — the build that produced it is the version
of the data — and reports only ever read it.

This notebook covers the whole report API. Each individual report has
its own notebook in this folder.

## What changed from 4.2.0

| | 4.2.0 | 4.3.0 |
| --- | --- | --- |
| data | PostgreSQL, or a parquet bundle through an ORM bridge | the bundle, read natively |
| `bf.report.run()` returns | a DataFrame | a `ReportResult` |
| provenance | lost on export | written beside the file |

There is no relational path any more: a report reads a bundle or it does
not run.

### 1. Open a bundle

In [ ]:
from pathlib import Path

from biofilter import Biofilter

# A bundle is a directory. Point at the directory, not at its tables/.
# Leave as None to use `[database] bundle` from .biofilter.toml.
BUNDLE = None

bf = Biofilter(bundle=BUNDLE, debug_mode=False) if BUNDLE else Biofilter(debug_mode=False)

# Results land here whatever directory the kernel was started in — VS Code
# and Jupyter disagree about that, and a bare filename ends up wherever
# they landed. The project root is the folder holding .biofilter.toml.
_root = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".biofilter.toml").is_file()),
    Path.cwd(),
)
OUTPUT_DIR = _root / "notebooks" / "templates" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

bf

On the command line the same thing is `--bundle`:

```bash
biofilter --bundle /path/to/bundles/20260914 report list
```

Or put the path in `.biofilter.toml` once and drop the argument:

```toml
[database]
bundle = "./biofilter_data/bundles/20260914"
```

A relative path there is relative to the config file, not to where you
are, so it works from a notebook in a subdirectory. With that set,
`Biofilter()` and `biofilter report run ...` both find it, and
`biofilter config show` prints which bundle is in effect.

### 2. What reports exist

In [ ]:
import pandas as pd

reports = bf.report.list()
df = pd.DataFrame(reports)

print(f"{len(df)} reports")
df[["name", "description"]]

Every report reads the bundle. The rewrite finished in 4.3.0 and the
frozen relational layer was deleted with it, so this list is the whole
catalogue — there is no second set waiting somewhere else.

The names start with what the report does: `resolve_`, `annotate_`,
`expand_`, `pair_`, `aggregate_`, `platform_`.

### 3. Ask a report about itself

In [ ]:
report_name = "annotate_gene"

print("columns:")
print(bf.report.available_columns(report_name))

print("\nexample input:")
print(bf.report.example_input(report_name))

In [ ]:
print(bf.report.explain(report_name))

### 4. Run one

In [ ]:
result = bf.report.run(report_name, input_data=["TP53", "BRCA1"])

# Native reports return a ReportResult, not a DataFrame.
print(type(result).__name__)
print(f"{result.num_rows} rows")

df = result.to_pandas()
df[["input_value", "gene_symbol", "hgnc_id", "chromosome", "status"]]

### 5. Provenance — which bundle produced this

`entity_id` and `variant_id` are **scoped to one bundle**. The same
integer means a different gene in the next build, and a stale id still
resolves — to the wrong row. Carrying the bundle id alongside the data is
what makes that detectable.

In [ ]:
result.provenance

### 6. Export

In [ ]:
# CSV, with a genes.csv.provenance.json written beside it.
written = result.write(OUTPUT_DIR / "genes.csv")
for path in written:
    print(path)

In [ ]:
# Parquet instead: the provenance travels inside the file's metadata,
# and list columns stay real lists rather than JSON strings.
result.write(OUTPUT_DIR / "genes.parquet")

### 7. Reading a result someone else produced

The sidecar is what lets you answer "where did this come from" months
later, and `build_record.json` in the bundle has the per-source detail
behind that id — which DTP, which version, which source URL.

In [ ]:
import json

with open(OUTPUT_DIR / "genes.csv.provenance.json") as fh:
    print(json.dumps(json.load(fh), indent=2))

### 8. Putting a result down and picking it up

`write()` **exports**: CSV to open elsewhere, parquet for size. It
flattens nested columns so a spreadsheet can hold them, which is lossy on
purpose — a list of aliases becomes a JSON string.

`save()` / `load()` is the other job: lose nothing, and stay usable
later. It writes a **directory**, not a file, because a result can carry
more than one table:

```
runs/genes/
├── manifest.json          provenance, and what tables are here
└── tables/
    └── result.parquet
```

In [ ]:
from biofilter.modules.report.result import ReportResult

saved = result.save(OUTPUT_DIR / "runs" / "genes", overwrite=True)
back = ReportResult.load(saved)

print("identical:", back.table.equals(result.table))
print("report   :", back.provenance["report"])
print("params   :", back.provenance["params"])

A result does not need the bundle that produced it, and does not go
looking. Whether that bundle is still on disk is recorded, because a
result outliving its bundle is the normal case and the reason to save
one — the rows are unchanged either way, and `bundle_id` names the build
whether or not the path still resolves.

In [ ]:
back.provenance["source_bundle"]

**Some reports return more than one table.** When a report's answer is
genuinely two shapes it says so, rather than flattening them into one or
writing the second out as a file:

```python
bins = bf.report.run("aggregate_cohort_variants", cohort_file="...",
                     output_grain="bins")
bins.table                            # one row per (bin, sample)
bins.extra_tables["variant_to_bin"]   # what each bin is made of
```

And because a saved result is laid out like a bundle, the reader
Biofilter already has opens it — so reusing one usually means querying
it, not loading it back into Python.

In [ ]:
from biofilter.modules.report import Bundle

with Bundle.open(saved) as opened:
    print("tables:", sorted(opened.tables))
    display(opened.con.execute(
        "SELECT * FROM result LIMIT 3"
    ).to_arrow_table().to_pandas())

**`provenance["warnings"]` is always there.** An empty list means
nothing went wrong, never "nothing was collected" — a report that copes
with a problem in silence leaves nothing behind, so coping gets written
down.

In [ ]:
result.provenance["warnings"]

---

## Working directly, without the facade

Reports are the packaged questions. For an ad-hoc one, open the bundle
and write SQL — it is the same engine the reports use.

In [ ]:
from biofilter.modules.report import Bundle

with Bundle.open(BUNDLE or bf.core.db_uri.removeprefix('parquet://')) as bundle:
    print(f"bundle {bundle.bundle_id}, {len(bundle.tables)} tables")

    # Every table is a view; query it as SQL and get Arrow back.
    out = bundle.con.execute("""
        SELECT g.name AS gene_group, count(*) AS genes
        FROM gene_masters gm
        JOIN gene_group_memberships m ON m.gene_id = gm.id
        JOIN gene_groups g ON g.id = m.group_id
        GROUP BY 1 ORDER BY genes DESC LIMIT 10
    """).to_arrow_table()

display(out.to_pandas())

### What is in this bundle

`bundle.tables` is the manifest, resolved: one entry per logical table,
with the files behind it. A partitioned table is many files and one view.

In [ ]:
with Bundle.open(BUNDLE or bf.core.db_uri.removeprefix('parquet://')) as bundle:
    inventory = pd.DataFrame(
        [
            {
                "table": t.name,
                "rows": t.rows,
                "files": len(t.files),
                "branch": t.branch,
            }
            for t in bundle.tables.values()
        ]
    ).sort_values("rows", ascending=False)

inventory.head(15)